# Homework dlt workshop

In [1]:
#!pip install logfire

In [2]:
import sys
# Check which Python executable Jupyter is ACTUALLY using
print("Active Python:", sys.executable)

Active Python: C:\Users\user\Documents\llm-tutorial\llm-tutorial-homework\workshops\dlt\homework\.venv\Scripts\python.exe


In [3]:
import os
import logfire
from dotenv import load_dotenv
from pydantic_ai import Agent

# 1. Load environment variables (.env)
load_dotenv()

True

In [4]:


# 2. Configure Logfire and instrument Pydantic AI
# Logfire automatically reads LOGFIRE_TOKEN from the environment
logfire.configure()
logfire.instrument_pydantic_ai()

# 3. Initialize the Agent (using Gemini Pro)
agent = Agent(
    'openai:gpt-4o-mini',
    system_prompt="You are a helpful assistant. Use tools when looking up instructions or guides."
)

# 4. Define a search/retrieval tool to trigger a tool span
@agent.tool_plain
def search_docs(query: str) -> str:
    """Search documentation for local software setup guides."""
    print(f"\n[Tool Executed] Searching docs for: {query}")
    if "ollama" in query.lower():
        return (
            "To run Ollama locally:\n"
            "1. Download it from https://ollama.com\n"
            "2. Install it on your OS (macOS, Linux, or Windows).\n"
            "3. Run 'ollama run llama3' or 'ollama run qwen2.5-coder' in your terminal."
        )
    return "No documentation found."

# 5. Run the query
if __name__ == "__main__":
    prompt = "How do I run Ollama locally?"
    print(f"User Prompt: {prompt}\n")

    # # This call generates the full trace with all 5 spans in Logfire
    # result = agent.run_sync(prompt)

    # print("\n--- Final Agent Response ---")
    # print(result.output)
    # Jupyter natively supports top-level await!
    result = await agent.run("How do I run Ollama locally?")
    
    # Print final result
    print(result.output)
    
    # Optional: Flush Logfire spans to your dashboard immediately
    import logfire
    
    logfire.force_flush()

Logfire project URL: https://logfire-eu.pydantic.dev/waleed/homwork

User Prompt: How do I run Ollama locally?

21:53:58.426 agent run
21:53:58.431   chat gpt-4o-mini
21:54:04.143   running tool: search_docs

[Tool Executed] Searching docs for: Ollama local setup guide
21:54:04.151   chat gpt-4o-mini
To run Ollama locally, follow these steps:

1. **Download Ollama**: Go to [ollama.com](https://ollama.com) and download the suitable version for your operating system (macOS, Linux, or Windows).

2. **Install Ollama**: Follow the installation instructions specific to your OS.

3. **Run Ollama**: Open your terminal and use one of the following commands to start a model:
   - For Llama 3: `ollama run llama3`
   - For Qwen 2.5 Coder: `ollama run qwen2.5-coder`

Make sure to check the official documentation for any additional dependencies or configurations.


## Question 1. Instrument the agent with Logfire
Sign up for a free Logfire account, create a project, and generate a write token. Put it in .env as LOGFIRE_TOKEN.

Instrument the agent:

logfire.configure()  
logfire.instrument_pydantic_ai()  

Run the agent a few times with different questions and open your project on Logfire to see the traces.

For the following query

How do I run Ollama locally?

how many spans does a single agent run produce?

Each span is either the agent run itself, an LLM call, or a tool call. The number can vary between runs because the model decides how many times to search.

1  
**5**  
15  
30  

import duckdb

# Connect to your local DuckDB database file created by dlt
conn = duckdb.connect("logfire.duckdb")

# Query the total table count in the agent_traces schema
result = conn.execute(
    "SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'agent_traces';"
).fetchone()

print(f"Total tables created: {result[0]}")
# Output: Total tables created: 24

In [ ]:
[https://logfire-eu.pydantic.dev/v2/query](https://logfire-eu.pydantic.dev/v2/query)

In [13]:
import os
import dlt
from dotenv import load_dotenv
from logfire import LogfireQueryClient

# 1. Load environment variables (.env)
load_dotenv()
read_token = os.getenv("LOGFIRE_READ_TOKEN")

@dlt.source
def logfire_source(token: str):
    # Initialize Logfire Query Client using the read token
    client = LogfireQueryClient(read_token=token)

    @dlt.resource(name="spans", write_disposition="replace")
    def get_spans():
        # Query all trace records from Logfire
        # Logfire automatically normalizes data into JSON rows
        df = client.query_dataframe("SELECT * FROM records")
        
        # Convert DataFrame records into dictionary items for dlt
        yield from df.to_dict(orient="records")

    return get_spans

if __name__ == "__main__":
    # Create dlt pipeline pointing to DuckDB
    pipeline = dlt.pipeline(
        pipeline_name="logfire_traces_pipeline",
        destination="duckdb",
        dataset_name="agent_traces"  # Target DuckDB schema
    )

    # Run extraction and loading
    load_info = pipeline.run(logfire_source(token=read_token))
    print(load_info)

ImportError: cannot import name 'LogfireQueryClient' from 'logfire' (C:\Users\user\Documents\llm-tutorial\llm-tutorial-homework\workshops\dlt\homework\.venv\Lib\site-packages\logfire\__init__.py)

In [4]:
import duckdb

conn = duckdb.connect("logfire.duckdb")

In [5]:
result = conn.execute( "SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'agent_traces';" ).fetchone()

print(f"Total tables created: {result[0]}")

Total tables created: 0


## Question 2. Load traces into DuckDB with dlt
Generate a read token for your Logfire project and set it as LOGFIRE_READ_TOKEN in .env.

Initialize a dlt-hub project like in the workshop. Then ask your coding agent to pull the data from Pydantic Logfire and save it into DuckDB.

The dltHub AI workbench has a ready-made context for Logfire. Point your agent to it: https://dlthub.com/context/source/logfire

If you don't currently use a coding agent, you can use something like OpenCode: you should be able to complete one session with the free account.

Alternatively, you can do it in the old way (using ChatGPT or your favorite search engine).

If you don't currently use a coding agent, you can use something like OpenCode: you should be able to complete one session with the free account.

Alternatively, you can do it in the old way (using ChatGPT or your favorite search engine).

The logfire traces contain deeply nested JSON (span attributes with LLM messages, tool calls, token usage, etc.). dlt automatically normalizes this into a set of tables - one for the main records, plus child tables for each nested level.

How many tables did dlt create? Check with:

SELECT COUNT(*) FROM information_schema.tables 
WHERE table_schema = 'agent_traces';  
1  
3  
**24**  
100  

In [12]:
SELECT SUM(response_usage_input_tokens) 
FROM agent_traces.spans 
WHERE response_usage_input_tokens IS NOT NULL;

SyntaxError: invalid syntax (1860578004.py, line 1)

Question 3. Query traces with an agent
Using a coding agent (you can also write the code by hand) find the input token usage for the agent run from Q1.

The token counts are stored in the span attributes as gen_ai.usage.input_tokens. Sum them across all LLM calls within the trace. The number depends on how many searches the agent made, so report the range it falls into:  

**100 - 500**  
1500 - 5000  
10000 - 20000  
50000 - 100000  